# Frozen Rater → Final Classifier on Waterbirds-ERM embeddings

This notebook trains **only the final linear classifier** with a previously trained, completely frozen Rater.

The main switch is:

```python
RATING_MODE = "all"
```

or:

```python
RATING_MODE = "misclassified"
```

For `all`, every example receives a Rater-derived weight. For `misclassified`, correct examples receive `CORRECT_WEIGHT` (default `1.0`) and only currently wrong examples receive Rater-derived weights.

The loss is always

\[
L=\sum_i w_i CE_i
\]

with **no division by** `sum(weights)`.

The default Rater and Waterbirds-ERM cache paths are already filled with the paths you provided. The notebook also lists all available `final_rater.pt` checkpoints on Drive so you can choose a different frozen Rater easily.

In [1]:
# ============================================================
# 1. Mount Drive + imports + list available frozen Raters
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from datetime import datetime
import json, csv, random

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

DRIVE_ROOT = Path('/content/drive/MyDrive/evidential_alignment_colab')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

search_roots = [
    DRIVE_ROOT / 'rater_meta_only_runs',
    DRIVE_ROOT / 'rater_ea_gate_meta_runs',
    DRIVE_ROOT / 'rater_erm_minimal_runs',
    DRIVE_ROOT / 'rater_classwise_top_loss_runs'
]

rater_candidates = []
for root in search_roots:
    if not root.exists():
        continue
    for path in root.rglob('final_rater.pt'):
        try:
            ckpt = torch.load(path, map_location='cpu', weights_only=False)
            if isinstance(ckpt, dict) and 'rater_sd' in ckpt:
                fp = ckpt.get('erm_fingerprint') or ckpt.get('erm_checkpoint_fingerprint')
                rater_candidates.append({
                    'path': path,
                    'mtime': path.stat().st_mtime,
                    'fingerprint': fp,
                    'capacity': ckpt.get('rater_capacity', '?'),
                    'weighting': ckpt.get('weighting') or ckpt.get('rater_weighting'),
                    'temperature': ckpt.get('temperature') or ckpt.get('rater_temperature'),
                })
        except Exception:
            pass

rater_candidates = sorted(rater_candidates, key=lambda x: x['mtime'], reverse=True)
print(f'Found {len(rater_candidates)} frozen Rater checkpoint(s):\n')
for i, item in enumerate(rater_candidates):
    print(f'[{i}] {item["path"]}')
    print('    fingerprint =', item['fingerprint'])
    print('    capacity =', item['capacity'], '| weighting =', item['weighting'], '| temperature =', item['temperature'])
    print()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Device: cpu
Found 7 frozen Rater checkpoint(s):

[0] /content/drive/MyDrive/evidential_alignment_colab/rater_classwise_top_loss_runs/top10pct_per_class_all_sample_20261011_131220/final_rater.pt
    fingerprint = 1055b1e8de29
    capacity = medium | weighting = sigmoid | temperature = 1.0

[1] /content/drive/MyDrive/evidential_alignment_colab/rater_classwise_top_loss_runs/top30pct_per_class_all_sample_20261007_215941/final_rater.pt
    fingerprint = 1055b1e8de29
    capacity = medium | weighting = sigmoid | temperature = 1.0

[2] /content/drive/MyDrive/evidential_alignment_colab/rater_meta_only_runs/ea_gated_full_epoch_on_all_20261005_081758/final_rater.pt
    fingerprint = 1055b1e8de29
    capacity = medium | weighting = sigmoid | temperature = 1.0

[3] /content/drive/MyDrive/evidential_alignment_colab/rater_meta_only_runs/ea_gated_full_epoch_random_inner_202

In [2]:
# ============================================================
# 2. EXPERIMENT CONFIGURATION
# ============================================================

# You can paste a full checkpoint path, or use:
# RATER_CHECKPOINT = rater_candidates[INDEX]['path']
# RATER_CHECKPOINT = Path(
#     '/content/drive/MyDrive/evidential_alignment_colab/'
#     'rater_classwise_top_loss_runs/'
#     'top30pct_per_class_all_sample_20261007_215941/'
#     'final_rater.pt'
# )

RATER_CHECKPOINT = Path(
    '/content/drive/MyDrive/evidential_alignment_colab/'
    'rater_classwise_top_loss_runs/'
    'top10pct_per_class_all_sample_20261011_131220/'
    'final_rater.pt'
)
EMBEDDING_CACHE_ROOT = Path(
    '/content/drive/MyDrive/evidential_alignment_colab/'
    'embeddings_waterbirds_erm/waterbirds/'
    'resnet50_waterbirds_erm_1055b1e8de29/resolution_224'
)
EXPECTED_ERM_FINGERPRINT = '1055b1e8de29'

# ------------------------------------------------------------
# MAIN SWITCH
# ------------------------------------------------------------
# "all"           -> every sample gets a Rater-derived weight
# "misclassified" -> correct=CORRECT_WEIGHT; wrong=Rater weight
RATING_MODE = 'all'

# Same final-classifier split setup as your previous notebook.
SPLIT_VAL = 0.6
SEED = 2

# Final classifier hyperparameters.
BATCH_SIZE = 128
NUM_WORKERS = 2
FINAL_EPOCHS = 50
FINAL_LR = 1e-2
FINAL_MOMENTUM = 0.9
FINAL_WEIGHT_DECAY = 1e-4

# Rater -> weight hyperparameters.
# NORMALIZATION_MODE: 'global', 'batch', or 'none'
NORMALIZATION_MODE = 'global'
# WEIGHT_TRANSFORM: 'sigmoid' or 'softmax'
WEIGHT_TRANSFORM = 'sigmoid'
TEMPERATURE = 1.0
WEIGHT_SCALE = 1.0
CORRECT_WEIGHT = 1.0  # used only for RATING_MODE='misclassified'

# Optional ordinary CE added beside weighted CE. 0 = weighted CE only.
AUX_CE_COEFF = 0.0

# Model selection and plotting.
SELECTION_METRIC = 'wga'  # 'wga' or 'accuracy'
PLOT_EVERY = 1

assert RATING_MODE in {'all', 'misclassified'}
assert NORMALIZATION_MODE in {'global', 'batch', 'none'}
assert WEIGHT_TRANSFORM in {'sigmoid', 'softmax'}
assert SELECTION_METRIC in {'wga', 'accuracy'}
assert TEMPERATURE > 0
assert RATER_CHECKPOINT.exists(), RATER_CHECKPOINT
assert EMBEDDING_CACHE_ROOT.exists(), EMBEDDING_CACHE_ROOT

RUNS_ROOT = DRIVE_ROOT / 'frozen_rater_final_runs_configurable'
RUN_NAME = (
    f'rating_{RATING_MODE}_norm_{NORMALIZATION_MODE}_{WEIGHT_TRANSFORM}'
    f'_temp_{TEMPERATURE}_lr_{FINAL_LR}_splitval_{SPLIT_VAL}_'
    f'{datetime.now().strftime("%Y%m%d_%H%M%S")}'
)
RUN_ROOT = RUNS_ROOT / RUN_NAME
PLOTS_ROOT = RUN_ROOT / 'plots'
RUN_ROOT.mkdir(parents=True, exist_ok=False)
PLOTS_ROOT.mkdir(parents=True, exist_ok=True)
LOG_PATH = RUN_ROOT / 'log.txt'
HISTORY_PATH = RUN_ROOT / 'history.csv'

print('Frozen Rater:', RATER_CHECKPOINT)
print('Embedding cache:', EMBEDDING_CACHE_ROOT)
print('RATING_MODE:', RATING_MODE)
print('Run directory:', RUN_ROOT)

Frozen Rater: /content/drive/MyDrive/evidential_alignment_colab/rater_classwise_top_loss_runs/top10pct_per_class_all_sample_20261011_131220/final_rater.pt
Embedding cache: /content/drive/MyDrive/evidential_alignment_colab/embeddings_waterbirds_erm/waterbirds/resnet50_waterbirds_erm_1055b1e8de29/resolution_224
RATING_MODE: all
Run directory: /content/drive/MyDrive/evidential_alignment_colab/frozen_rater_final_runs_configurable/rating_all_norm_global_sigmoid_temp_1.0_lr_0.01_splitval_0.6_20261011_141149


In [3]:
# ============================================================
# 3. Reproducibility + logging + configuration dump
# ============================================================

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

def log(msg=''):
    msg = str(msg)
    print(msg, flush=True)
    with open(LOG_PATH, 'a', encoding='utf-8') as f:
        f.write(msg + '\n')

config = {
    'rater_checkpoint': str(RATER_CHECKPOINT),
    'embedding_cache_root': str(EMBEDDING_CACHE_ROOT),
    'expected_erm_fingerprint': EXPECTED_ERM_FINGERPRINT,
    'rating_mode': RATING_MODE,
    'split_val': SPLIT_VAL,
    'seed': SEED,
    'batch_size': BATCH_SIZE,
    'final_epochs': FINAL_EPOCHS,
    'final_lr': FINAL_LR,
    'final_momentum': FINAL_MOMENTUM,
    'final_weight_decay': FINAL_WEIGHT_DECAY,
    'normalization_mode': NORMALIZATION_MODE,
    'weight_transform': WEIGHT_TRANSFORM,
    'temperature': TEMPERATURE,
    'weight_scale': WEIGHT_SCALE,
    'correct_weight': CORRECT_WEIGHT,
    'aux_ce_coeff': AUX_CE_COEFF,
    'selection_metric': SELECTION_METRIC,
    'plot_every': PLOT_EVERY,
}
with open(RUN_ROOT / 'config.json', 'w', encoding='utf-8') as f:
    json.dump(config, f, indent=2)

log('=' * 100)
log('FROZEN RATER FINAL CLASSIFIER')
log('=' * 100)
log(json.dumps(config, indent=2))
log('=' * 100)

FROZEN RATER FINAL CLASSIFIER
{
  "rater_checkpoint": "/content/drive/MyDrive/evidential_alignment_colab/rater_classwise_top_loss_runs/top10pct_per_class_all_sample_20261011_131220/final_rater.pt",
  "embedding_cache_root": "/content/drive/MyDrive/evidential_alignment_colab/embeddings_waterbirds_erm/waterbirds/resnet50_waterbirds_erm_1055b1e8de29/resolution_224",
  "expected_erm_fingerprint": "1055b1e8de29",
  "rating_mode": "all",
  "split_val": 0.6,
  "seed": 2,
  "batch_size": 128,
  "final_epochs": 50,
  "final_lr": 0.01,
  "final_momentum": 0.9,
  "final_weight_decay": 0.0001,
  "normalization_mode": "global",
  "weight_transform": "sigmoid",
  "temperature": 1.0,
  "weight_scale": 1.0,
  "correct_weight": 1.0,
  "aux_ce_coeff": 0.0,
  "selection_metric": "wga",
  "plot_every": 1
}


In [4]:
# ============================================================
# 4. Load Waterbirds-ERM embedding splits
# ============================================================

class IndexedEmbeddingDataset(Dataset):
    def __init__(self, embeddings, labels, groups, attrs, split_name):
        self.embeddings = embeddings.float()
        self.labels = labels.long()
        self.groups = groups.long()
        self.attrs = attrs.long()
        self.split_name = split_name

    def __len__(self):
        return int(self.labels.shape[0])

    def __getitem__(self, idx):
        return idx, self.embeddings[idx], self.labels[idx], self.groups[idx], self.attrs[idx]


def _tok(x):
    return str(x).replace('.', 'p')


def split_path(split_name):
    if split_name in {'train_no_aug', 'val', 'test'}:
        return EMBEDDING_CACHE_ROOT / f'{split_name}.pt'
    if split_name in {'val_subset1', 'val_subset2'}:
        return EMBEDDING_CACHE_ROOT / (
            f'{split_name}_splittrain_1p0_splitval_{_tok(SPLIT_VAL)}_seed_{SEED}.pt'
        )
    raise ValueError(split_name)


def ensure_val_subsets():
    """Create requested split_val subsets directly from saved val.pt if absent."""
    p1 = split_path('val_subset1')
    p2 = split_path('val_subset2')
    if p1.exists() and p2.exists():
        log('[Embeddings] requested val subsets already exist.')
        return

    full_path = split_path('val')
    payload = torch.load(full_path, map_location='cpu', weights_only=False)
    n = int(payload['labels'].shape[0])

    rng = np.random.RandomState(SEED)
    indices = np.arange(n)
    rng.shuffle(indices)
    n1 = int(n * SPLIT_VAL)

    subset_indices = {
        'val_subset1': torch.as_tensor(indices[:n1], dtype=torch.long),
        'val_subset2': torch.as_tensor(indices[n1:], dtype=torch.long),
    }

    for split_name, idx in subset_indices.items():
        out_path = split_path(split_name)
        subset = {
            'embeddings': payload['embeddings'][idx].clone(),
            'labels': payload['labels'][idx].clone(),
            'groups': payload['groups'][idx].clone(),
            'attrs': payload['attrs'][idx].clone(),
            'cache_metadata': dict(payload.get('cache_metadata', {})),
        }
        subset['cache_metadata'].update({
            'split': split_name,
            'split_train': 1.0,
            'split_val': float(SPLIT_VAL),
            'seed': int(SEED),
            'derived_from_full_val_cache': True,
        })
        torch.save(subset, out_path)
        log(f'[Embeddings] created {split_name}: n={len(idx)}')


ensure_val_subsets()


def load_split(split_name):
    path = split_path(split_name)
    payload = torch.load(path, map_location='cpu', weights_only=False)
    z = payload['embeddings']
    assert z.ndim == 2 and z.shape[1] == 2048, (split_name, z.shape)

    metadata = payload.get('cache_metadata', {})
    representation = metadata.get('representation_source')
    if representation and representation != 'waterbirds_erm_checkpoint':
        raise RuntimeError(f'{split_name} is not a Waterbirds-ERM embedding cache.')

    fp = metadata.get('erm_checkpoint_fingerprint') or metadata.get('erm_fingerprint')
    if fp and fp != EXPECTED_ERM_FINGERPRINT:
        raise RuntimeError(f'{split_name} fingerprint mismatch: {fp}')

    ds = IndexedEmbeddingDataset(
        payload['embeddings'], payload['labels'], payload['groups'], payload['attrs'], split_name
    )
    log(f'[Embeddings] {split_name}: {len(ds)} samples')
    return ds


# Same roles as the previous final-classifier notebook.
calibration_split = 'val_subset1'
selection_split = 'val_subset2'
test_split = 'test'

calibration_dataset = load_split(calibration_split)
selection_dataset = load_split(selection_split)
test_dataset = load_split(test_split)

for name, ds in [('calibration', calibration_dataset), ('selection', selection_dataset), ('test', test_dataset)]:
    log('')
    log(f'{name}: n={len(ds)}')
    g = ds.groups.numpy()
    for gid in sorted(np.unique(g)):
        log(f'  g{gid}: {int((g == gid).sum())}')

[Embeddings] requested val subsets already exist.
[Embeddings] val_subset1: 719 samples
[Embeddings] val_subset2: 480 samples
[Embeddings] test: 5794 samples

calibration: n=719
  g0: 284
  g1: 280
  g2: 78
  g3: 77

selection: n=480
  g0: 183
  g1: 186
  g2: 55
  g3: 56

test: n=5794
  g0: 2255
  g1: 2255
  g2: 642
  g3: 642


In [5]:
# ============================================================
# 5. Load and COMPLETELY FREEZE the selected Rater
# ============================================================

class EmbeddingRaterSmall(nn.Module):
    def __init__(self, input_dim=2048):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(input_dim),
            nn.Linear(input_dim, 256),
            nn.ReLU(inplace=True),
            nn.Linear(256, 64),
            nn.ReLU(inplace=True),
            nn.Linear(64, 1),
        )
    def forward(self, z):
        return self.net(z).squeeze(-1)


class EmbeddingRaterMedium(nn.Module):
    def __init__(self, input_dim=2048):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(input_dim),
            nn.Linear(input_dim, 512),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(512, 256),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(256, 64),
            nn.GELU(),
            nn.Linear(64, 1),
        )
    def forward(self, z):
        return self.net(z).squeeze(-1)


checkpoint = torch.load(RATER_CHECKPOINT, map_location='cpu', weights_only=False)
state_dict = checkpoint['rater_sd']
capacity = str(checkpoint.get('rater_capacity', '')).lower()

if capacity not in {'small', 'medium'}:
    first = state_dict.get('net.1.weight')
    if first is not None and first.shape[0] == 256:
        capacity = 'small'
    elif first is not None and first.shape[0] == 512:
        capacity = 'medium'
    else:
        raise RuntimeError('Could not infer Rater architecture.')

rater = EmbeddingRaterSmall(2048) if capacity == 'small' else EmbeddingRaterMedium(2048)
rater.load_state_dict(state_dict)
rater.to(DEVICE)
rater.eval()
for p in rater.parameters():
    p.requires_grad_(False)
assert not any(p.requires_grad for p in rater.parameters())

checkpoint_fp = checkpoint.get('erm_fingerprint') or checkpoint.get('erm_checkpoint_fingerprint')
if checkpoint_fp and checkpoint_fp != EXPECTED_ERM_FINGERPRINT:
    raise RuntimeError(
        f'Rater fingerprint {checkpoint_fp} != embedding fingerprint {EXPECTED_ERM_FINGERPRINT}'
    )

log('')
log('Frozen Rater loaded:')
log(f'  path = {RATER_CHECKPOINT}')
log(f'  capacity = {capacity}')
log(f'  ERM fingerprint = {checkpoint_fp}')
log(f'  Rater training weighting = {checkpoint.get("weighting") or checkpoint.get("rater_weighting")}')
log(f'  Rater training temperature = {checkpoint.get("temperature") or checkpoint.get("rater_temperature")}')
log('  trainable Rater parameters = 0')


Frozen Rater loaded:
  path = /content/drive/MyDrive/evidential_alignment_colab/rater_classwise_top_loss_runs/top10pct_per_class_all_sample_20261011_131220/final_rater.pt
  capacity = medium
  ERM fingerprint = 1055b1e8de29
  Rater training weighting = sigmoid
  Rater training temperature = 1.0
  trainable Rater parameters = 0


In [6]:
# ============================================================
# 6. Precompute frozen-Rater raw scores once
# ============================================================

@torch.no_grad()
def precompute_raw_scores(dataset):
    loader = DataLoader(
        dataset, batch_size=512, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=True,
    )
    scores = torch.empty(len(dataset), dtype=torch.float32)
    rater.eval()

    for idx, z, _, _, _ in loader:
        z = z.to(DEVICE, non_blocking=True)
        scores[idx.long()] = rater(z).detach().cpu()

    return scores


calibration_raw_scores = precompute_raw_scores(calibration_dataset)
selection_raw_scores = precompute_raw_scores(selection_dataset)
test_raw_scores = precompute_raw_scores(test_dataset)

GLOBAL_RATER_MEAN = float(calibration_raw_scores.mean())
GLOBAL_RATER_STD = float(calibration_raw_scores.std(unbiased=False))

log('')
log(f'Global calibration Rater mean = {GLOBAL_RATER_MEAN:.8f}')
log(f'Global calibration Rater std  = {GLOBAL_RATER_STD:.8f}')

# Raw Rater distribution by Waterbirds group.
fig, ax = plt.subplots(figsize=(10, 7))
groups = calibration_dataset.groups.numpy()
raw = calibration_raw_scores.numpy()
for gid in sorted(np.unique(groups)):
    mask = groups == gid
    ax.hist(raw[mask], bins=40, density=True, alpha=0.45, label=f'g{gid} (n={int(mask.sum())})')
ax.set_xlabel('Frozen Rater raw score')
ax.set_ylabel('Density')
ax.set_title('Frozen Rater raw scores on calibration set')
ax.grid(True, linestyle=':', alpha=0.3)
ax.legend()
fig.tight_layout()
fig.savefig(PLOTS_ROOT / 'calibration_raw_rater_scores_by_group.png', dpi=160, bbox_inches='tight')
plt.close(fig)

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)



Global calibration Rater mean = -0.06717836
Global calibration Rater std  = 0.05202202


In [7]:
# ============================================================
# 7. Configurable Rater-to-weight rule
# ============================================================

def normalize_scores(raw_scores):
    if NORMALIZATION_MODE == 'global':
        return (raw_scores - GLOBAL_RATER_MEAN) / (GLOBAL_RATER_STD + 1e-6)

    if NORMALIZATION_MODE == 'batch':
        mean = raw_scores.mean()
        std = raw_scores.std(unbiased=False)
        return (raw_scores - mean) / (std + 1e-6)

    if NORMALIZATION_MODE == 'none':
        return raw_scores

    raise ValueError(NORMALIZATION_MODE)


def transform_scores(normalized_scores):
    if WEIGHT_TRANSFORM == 'sigmoid':
        return WEIGHT_SCALE * torch.sigmoid(normalized_scores / TEMPERATURE)

    if WEIGHT_TRANSFORM == 'softmax':
        centered = normalized_scores - normalized_scores.mean()
        return WEIGHT_SCALE * torch.softmax(centered / TEMPERATURE, dim=0)

    raise ValueError(WEIGHT_TRANSFORM)


@torch.no_grad()
def compute_training_weights(batch_indices, logits, y):
    """
    RATING_MODE='all':
        every sample gets a transformed Rater weight.

    RATING_MODE='misclassified':
        correct samples get CORRECT_WEIGHT;
        wrong samples get transformed Rater weights.
    """
    idx_cpu = batch_indices.detach().cpu().long()
    raw_all = calibration_raw_scores[idx_cpu].to(logits.device)

    pred = logits.argmax(dim=1)
    correct_mask = pred.eq(y)
    wrong_mask = ~correct_mask

    if RATING_MODE == 'all':
        weights = transform_scores(normalize_scores(raw_all))
        return weights, correct_mask, wrong_mask, raw_all

    # Misclassified-only mode.
    weights = torch.full(
        (y.shape[0],), float(CORRECT_WEIGHT),
        device=logits.device, dtype=logits.dtype,
    )

    if bool(wrong_mask.any()):
        raw_wrong = raw_all[wrong_mask]

        if NORMALIZATION_MODE == 'global':
            normalized_wrong = (raw_wrong - GLOBAL_RATER_MEAN) / (GLOBAL_RATER_STD + 1e-6)
        elif NORMALIZATION_MODE == 'batch':
            normalized_wrong = (
                raw_wrong - raw_wrong.mean()
            ) / (
                raw_wrong.std(unbiased=False) + 1e-6
            )
        elif NORMALIZATION_MODE == 'none':
            normalized_wrong = raw_wrong
        else:
            raise ValueError(NORMALIZATION_MODE)

        wrong_weights = transform_scores(normalized_wrong)
        wrong_idx = torch.nonzero(wrong_mask, as_tuple=False).squeeze(1)
        weights = weights.index_copy(0, wrong_idx, wrong_weights)

    return weights, correct_mask, wrong_mask, raw_all

In [8]:
# ============================================================
# 8. Fresh final classifier + evaluation + plotting helpers
# ============================================================

class FinalLinearClassifier(nn.Module):
    def __init__(self, input_dim=2048, num_classes=2):
        super().__init__()
        self.linear = nn.Linear(input_dim, num_classes)

    def forward(self, z):
        return self.linear(z)


@torch.no_grad()
def evaluate_classifier(model, dataset):
    loader = DataLoader(
        dataset, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=True,
    )
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_n = 0
    group_correct = {}
    group_total = {}

    for _, z, y, g, _ in loader:
        z = z.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)
        g = g.to(DEVICE, non_blocking=True)

        logits = model(z)
        pred = logits.argmax(dim=1)
        total_loss += float(F.cross_entropy(logits, y, reduction='sum').item())
        total_correct += int(pred.eq(y).sum().item())
        total_n += int(y.numel())

        for gid_t in torch.unique(g):
            gid = int(gid_t.item())
            mask = g.eq(gid_t)
            group_correct[gid] = group_correct.get(gid, 0) + int(pred[mask].eq(y[mask]).sum().item())
            group_total[gid] = group_total.get(gid, 0) + int(mask.sum().item())

    group_accuracy = {
        gid: group_correct[gid] / group_total[gid]
        for gid in sorted(group_total)
    }

    return {
        'loss': total_loss / max(total_n, 1),
        'accuracy': total_correct / max(total_n, 1),
        'group_accuracy': group_accuracy,
        'wga': min(group_accuracy.values()) if group_accuracy else float('nan'),
    }


def selection_value(metrics):
    return metrics['wga'] if SELECTION_METRIC == 'wga' else metrics['accuracy']


def save_epoch_weight_plots(epoch, weights, groups, correctness):
    weights = np.asarray(weights)
    groups = np.asarray(groups)
    correctness = np.asarray(correctness) > 0.5

    # By Waterbirds group.
    fig, ax = plt.subplots(figsize=(11, 7))
    for gid in sorted(np.unique(groups)):
        mask = groups == gid
        ax.hist(weights[mask], bins=40, density=True, alpha=0.45,
                label=f'g{gid} (n={int(mask.sum())})')
    ax.set_xlabel('Actual training weight')
    ax.set_ylabel('Density')
    ax.set_title(f'Epoch {epoch}: actual weights by group | mode={RATING_MODE}')
    ax.grid(True, linestyle=':', alpha=0.3)
    ax.legend()
    fig.tight_layout()
    fig.savefig(PLOTS_ROOT / f'epoch_{epoch:03d}_weights_by_group.png', dpi=160, bbox_inches='tight')
    plt.close(fig)

    # By current classifier correctness.
    fig, ax = plt.subplots(figsize=(10, 7))
    if bool(correctness.any()):
        ax.hist(weights[correctness], bins=40, density=True, alpha=0.5,
                label=f'correct (n={int(correctness.sum())})')
    wrong = ~correctness
    if bool(wrong.any()):
        ax.hist(weights[wrong], bins=40, density=True, alpha=0.5,
                label=f'wrong (n={int(wrong.sum())})')
    ax.set_xlabel('Actual training weight')
    ax.set_ylabel('Density')
    ax.set_title(f'Epoch {epoch}: weights by correctness | mode={RATING_MODE}')
    ax.grid(True, linestyle=':', alpha=0.3)
    ax.legend()
    fig.tight_layout()
    fig.savefig(PLOTS_ROOT / f'epoch_{epoch:03d}_weights_correct_vs_wrong.png', dpi=160, bbox_inches='tight')
    plt.close(fig)

In [9]:
# ============================================================
# 9. Train the final classifier with the frozen Rater
# ============================================================

set_seed(SEED)

# Fresh random Linear(2048, 2), matching the previous final-classifier notebook.
final_classifier = FinalLinearClassifier(2048, 2).to(DEVICE)

optimizer = torch.optim.SGD(
    final_classifier.parameters(),
    lr=FINAL_LR,
    momentum=FINAL_MOMENTUM,
    weight_decay=FINAL_WEIGHT_DECAY,
)

train_loader = DataLoader(
    calibration_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

best_value = -float('inf')
best_epoch = None
best_state = None
best_selection_metrics = None

oracle_test_wga = -float('inf')
oracle_test_epoch = None
history = []

for epoch in range(1, FINAL_EPOCHS + 1):
    final_classifier.train()

    weighted_loss_sum = 0.0
    ordinary_ce_sum = 0.0
    train_correct = 0
    train_n = 0
    n_batches = 0

    epoch_weights = []
    epoch_groups = []
    epoch_correctness = []

    for batch_indices, z, y, g, _ in train_loader:
        z = z.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)
        g = g.to(DEVICE, non_blocking=True)

        logits = final_classifier(z)

        weights, correct_mask, wrong_mask, raw_scores = compute_training_weights(
            batch_indices=batch_indices,
            logits=logits,
            y=y,
        )

        per_sample_ce = F.cross_entropy(logits, y, reduction='none')
        weighted_ce = (per_sample_ce * weights).sum()
        ordinary_ce = per_sample_ce.mean()
        loss = weighted_ce + AUX_CE_COEFF * ordinary_ce

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_correct += int(correct_mask.sum().item())
        train_n += int(y.numel())
        weighted_loss_sum += float(weighted_ce.item())
        ordinary_ce_sum += float(ordinary_ce.item())
        n_batches += 1

        epoch_weights.append(weights.detach().cpu())
        epoch_groups.append(g.detach().cpu())
        epoch_correctness.append(correct_mask.float().detach().cpu())

    weights_np = torch.cat(epoch_weights).numpy()
    groups_np = torch.cat(epoch_groups).numpy()
    correctness_np = torch.cat(epoch_correctness).numpy()

    selection_metrics = evaluate_classifier(final_classifier, selection_dataset)
    test_metrics = evaluate_classifier(final_classifier, test_dataset)

    current_value = selection_value(selection_metrics)
    if current_value > best_value:
        best_value = float(current_value)
        best_epoch = int(epoch)
        best_state = {
            key: value.detach().cpu().clone()
            for key, value in final_classifier.state_dict().items()
        }
        best_selection_metrics = dict(selection_metrics)

    if test_metrics['wga'] > oracle_test_wga:
        oracle_test_wga = float(test_metrics['wga'])
        oracle_test_epoch = int(epoch)

    row = {
        'epoch': epoch,
        'weighted_train_loss_per_batch': weighted_loss_sum / max(n_batches, 1),
        'ordinary_train_ce_per_batch': ordinary_ce_sum / max(n_batches, 1),
        'train_accuracy': train_correct / max(train_n, 1),
        'weight_mean': float(weights_np.mean()),
        'weight_std': float(weights_np.std()),
        'weight_min': float(weights_np.min()),
        'weight_max': float(weights_np.max()),
        'selection_accuracy': selection_metrics['accuracy'],
        'selection_wga': selection_metrics['wga'],
        'test_accuracy': test_metrics['accuracy'],
        'test_wga': test_metrics['wga'],
    }
    history.append(row)

    log(
        f'[epoch {epoch:03d}/{FINAL_EPOCHS}] '
        f'train_acc={100*row["train_accuracy"]:.2f}% | '
        f'w_mean={row["weight_mean"]:.4f} | '
        f'val_acc={100*row["selection_accuracy"]:.2f}% | '
        f'val_WGA={100*row["selection_wga"]:.2f}% | '
        f'test_acc={100*row["test_accuracy"]:.2f}% | '
        f'test_WGA={100*row["test_wga"]:.2f}%'
    )

    if epoch == 1 or epoch % PLOT_EVERY == 0 or epoch == FINAL_EPOCHS:
        save_epoch_weight_plots(epoch, weights_np, groups_np, correctness_np)


if best_state is None:
    raise RuntimeError('No final-classifier state selected.')

final_classifier.load_state_dict(best_state)
final_classifier.to(DEVICE)
final_classifier.eval()
restored_test_metrics = evaluate_classifier(final_classifier, test_dataset)

# Save selected classifier.
torch.save(
    {
        'model_sd': final_classifier.state_dict(),
        'selected_epoch': best_epoch,
        'selection_metric': SELECTION_METRIC,
        'selection_metrics': best_selection_metrics,
        'test_metrics': restored_test_metrics,
        'rating_mode': RATING_MODE,
        'normalization_mode': NORMALIZATION_MODE,
        'weight_transform': WEIGHT_TRANSFORM,
        'temperature': TEMPERATURE,
        'rater_checkpoint': str(RATER_CHECKPOINT),
        'embedding_cache_root': str(EMBEDDING_CACHE_ROOT),
    },
    RUN_ROOT / 'best_final_classifier.pt',
)

# History CSV.
with open(HISTORY_PATH, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=list(history[0].keys()))
    writer.writeheader()
    writer.writerows(history)

log('')
log('=' * 100)
log(f'SELECTED EPOCH = {best_epoch}')
log(f'selection accuracy = {100*best_selection_metrics["accuracy"]:.2f}%')
log(f'selection WGA      = {100*best_selection_metrics["wga"]:.2f}%')
log(f'restored test acc  = {100*restored_test_metrics["accuracy"]:.2f}%')
log(f'restored test WGA  = {100*restored_test_metrics["wga"]:.2f}%')
log(f'test-oracle WGA    = {100*oracle_test_wga:.2f}% at epoch {oracle_test_epoch}')
log('=' * 100)

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


[epoch 001/50] train_acc=70.38% | w_mean=0.5036 | val_acc=85.62% | val_WGA=27.27% | test_acc=86.18% | test_WGA=29.91%
[epoch 002/50] train_acc=92.49% | w_mean=0.5036 | val_acc=91.25% | val_WGA=84.95% | test_acc=89.94% | test_WGA=81.82%
[epoch 003/50] train_acc=90.26% | w_mean=0.5036 | val_acc=95.00% | val_WGA=81.82% | test_acc=94.98% | test_WGA=88.16%
[epoch 004/50] train_acc=95.41% | w_mean=0.5036 | val_acc=93.96% | val_WGA=74.55% | test_acc=95.13% | test_WGA=79.75%
[epoch 005/50] train_acc=94.99% | w_mean=0.5036 | val_acc=94.79% | val_WGA=80.00% | test_acc=95.55% | test_WGA=84.11%
[epoch 006/50] train_acc=96.24% | w_mean=0.5036 | val_acc=94.17% | val_WGA=83.64% | test_acc=95.29% | test_WGA=90.19%
[epoch 007/50] train_acc=95.97% | w_mean=0.5036 | val_acc=94.58% | val_WGA=81.82% | test_acc=95.62% | test_WGA=88.47%
[epoch 008/50] train_acc=97.36% | w_mean=0.5036 | val_acc=94.17% | val_WGA=78.18% | test_acc=95.60% | test_WGA=85.51%
[epoch 009/50] train_acc=97.64% | w_mean=0.5036 | val_ac

In [10]:
# ============================================================
# 10. Final curves + summary
# ============================================================

epochs = [row['epoch'] for row in history]

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(epochs, [row['selection_accuracy'] for row in history], label='selection accuracy')
ax.plot(epochs, [row['selection_wga'] for row in history], label='selection WGA')
ax.plot(epochs, [row['test_accuracy'] for row in history], label='test accuracy', alpha=0.75)
ax.plot(epochs, [row['test_wga'] for row in history], label='test WGA', alpha=0.75)
ax.axvline(best_epoch, linestyle='--', linewidth=1.5, label=f'selected epoch {best_epoch}')
ax.set_xlabel('Epoch')
ax.set_ylabel('Accuracy')
ax.set_title(f'Final classifier performance | rating mode={RATING_MODE}')
ax.grid(True, linestyle=':', alpha=0.3)
ax.legend()
fig.tight_layout()
fig.savefig(PLOTS_ROOT / 'final_classifier_accuracy_wga.png', dpi=160, bbox_inches='tight')
plt.close(fig)

fig, ax = plt.subplots(figsize=(10, 6))
mean_w = np.asarray([row['weight_mean'] for row in history])
std_w = np.asarray([row['weight_std'] for row in history])
ax.plot(epochs, mean_w, label='mean training weight')
ax.fill_between(epochs, mean_w - std_w, mean_w + std_w, alpha=0.2, label='±1 std')
ax.set_xlabel('Epoch')
ax.set_ylabel('Weight')
ax.set_title(f'Training-weight trajectory | rating mode={RATING_MODE}')
ax.grid(True, linestyle=':', alpha=0.3)
ax.legend()
fig.tight_layout()
fig.savefig(PLOTS_ROOT / 'training_weight_trajectory.png', dpi=160, bbox_inches='tight')
plt.close(fig)

summary = {
    'rating_mode': RATING_MODE,
    'rater_checkpoint': str(RATER_CHECKPOINT),
    'embedding_cache_root': str(EMBEDDING_CACHE_ROOT),
    'selected_epoch': best_epoch,
    'selection_metric': SELECTION_METRIC,
    'selection_metrics': best_selection_metrics,
    'restored_test_metrics': restored_test_metrics,
    'test_oracle_wga': oracle_test_wga,
    'test_oracle_epoch': oracle_test_epoch,
    'normalization_mode': NORMALIZATION_MODE,
    'weight_transform': WEIGHT_TRANSFORM,
    'temperature': TEMPERATURE,
    'weight_scale': WEIGHT_SCALE,
    'correct_weight': CORRECT_WEIGHT,
    'aux_ce_coeff': AUX_CE_COEFF,
}

with open(RUN_ROOT / 'summary.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2)

with open(RUN_ROOT / 'summary.txt', 'w', encoding='utf-8') as f:
    f.write('FROZEN RATER FINAL CLASSIFIER\n')
    f.write('=' * 80 + '\n')
    f.write(f'RATING_MODE = {RATING_MODE}\n')
    f.write(f'Rater = {RATER_CHECKPOINT}\n')
    f.write(f'Embeddings = {EMBEDDING_CACHE_ROOT}\n')
    f.write(f'Selected epoch = {best_epoch}\n')
    f.write(f'Selection accuracy = {best_selection_metrics["accuracy"]:.8f}\n')
    f.write(f'Selection WGA = {best_selection_metrics["wga"]:.8f}\n')
    f.write(f'Restored test accuracy = {restored_test_metrics["accuracy"]:.8f}\n')
    f.write(f'Restored test WGA = {restored_test_metrics["wga"]:.8f}\n')
    f.write(f'Test-oracle WGA = {oracle_test_wga:.8f} at epoch {oracle_test_epoch}\n')

print('Done.')
print('Run directory:')
print(RUN_ROOT)
print('Best classifier:')
print(RUN_ROOT / 'best_final_classifier.pt')
print('Plots:')
print(PLOTS_ROOT)

Done.
Run directory:
/content/drive/MyDrive/evidential_alignment_colab/frozen_rater_final_runs_configurable/rating_all_norm_global_sigmoid_temp_1.0_lr_0.01_splitval_0.6_20261011_141149
Best classifier:
/content/drive/MyDrive/evidential_alignment_colab/frozen_rater_final_runs_configurable/rating_all_norm_global_sigmoid_temp_1.0_lr_0.01_splitval_0.6_20261011_141149/best_final_classifier.pt
Plots:
/content/drive/MyDrive/evidential_alignment_colab/frozen_rater_final_runs_configurable/rating_all_norm_global_sigmoid_temp_1.0_lr_0.01_splitval_0.6_20261011_141149/plots


## Quick switches

**Rate every sample**:

```python
RATING_MODE = "all"
NORMALIZATION_MODE = "global"
WEIGHT_TRANSFORM = "sigmoid"
TEMPERATURE = 1.0
```

Then every calibration example gets a Rater-derived weight.

**Rate only misclassified samples**:

```python
RATING_MODE = "misclassified"
CORRECT_WEIGHT = 1.0
NORMALIZATION_MODE = "global"
WEIGHT_TRANSFORM = "sigmoid"
TEMPERATURE = 1.0
```

Then correct examples receive weight `1.0`, while wrong examples receive the Rater-derived weight.

To choose another Rater, use the list printed near the top and set either:

```python
RATER_CHECKPOINT = rater_candidates[INDEX]["path"]
```

or paste the full path directly.